Este módulo detalla la transición metodológica desde modelos de representación vectorial hacia matrices continuas (rasters), optimizadas para el análisis espacial homogéneo.

Primeramente se programó un algoritmo de rasterización mediante rasterio.features.rasterize sobre la capa de uso de suelo previamente depurada, asignándole un gradiente de aptitud numérica entera $[0, 5]$ indexado a la resistencia biofísica de cada cobertura (ej. asignando valor máximo 5 a la estepa por su viabilidad de instalación, y valor de exclusión 0 a los cuerpos de agua).

Por último se unificaron las 4 zonas del uso de suelo, proceso que se hizo repetidamente en este notebook debido a las limitacaiones de la RAM del programa, pero que al final terminó resultando.

In [ ]:
# CELDA 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELDA 2 — Verificar RAM disponible
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi        10Gi       2.0Mi       829Mi        11Gi
Swap:             0B          0B          0B


In [ ]:
# CELDA 3 — Imports y listar shapefiles
import geopandas as gpd
import pandas as pd
import glob
import gc

carpeta_raw = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw '
shapefiles = sorted(glob.glob(f'{carpeta_raw}/**/*.shp', recursive=True))
columnas_utiles = ['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'geometry']

print(f"Shapefiles encontrados: {len(shapefiles)}")
for s in shapefiles:
    print(f"  {s}")

Shapefiles encontrados: 4
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_este.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_2_oeste.shp
  /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_raw /12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3/12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_3.shp


In [ ]:
# CELDA 4 — Función robusta de carga con encoding
def cargar_zona_robusto(ruta_shp, columnas):
    for enc in ['cp1252', 'latin-1', 'utf-8']:
        gdf = gpd.read_file(ruta_shp, encoding=enc)[columnas].copy()
        if not gdf['USO'].astype(str).str.contains('Ã').any():
            print(f"  ✓ encoding correcto: {enc}")
            return gdf
    print(f"  ⚠️ ningún encoding limpió el texto, usando cp1252 de todas formas")
    return gpd.read_file(ruta_shp, encoding='cp1252')[columnas].copy()

zonas_livianas = []
for shp in shapefiles:
    nombre = shp.split('/')[-1]
    print(f"Cargando {nombre}...")
    gdf = cargar_zona_robusto(shp, columnas_utiles)
    zonas_livianas.append(gdf)
    gc.collect()

print(f"\n✅ {len(zonas_livianas)} zonas cargadas con encoding verificado")

Cargando 12_regi_n_de_magallanes_actualizaci_n_2017_2019_zona_1.shp...


KeyboardInterrupt: 

In [ ]:
# CELDA 5 — Unificar las 4 zonas
uso_suelo_magallanes = pd.concat(zonas_livianas, ignore_index=True)
uso_suelo_magallanes = gpd.GeoDataFrame(uso_suelo_magallanes, crs='EPSG:32719')

del zonas_livianas
gc.collect()

print(f"✅ Unificado: {len(uso_suelo_magallanes)} polígonos")
print(uso_suelo_magallanes['USO'].value_counts())

✅ Unificado: 286529 polígonos
USO
Bosques                                  111638
Praderas y Matorrales                    106399
Cuerpos de Agua                           32091
Áreas Desprovistas de Vegetación          14859
Humedales                                 14432
ÃƒÂreas Desprovistas de VegetaciÃƒÂ³n      3562
Nieves Eternas y Glaciares                 1205
Terrenos AgrÃƒÂ­colas                       807
ÃƒÂreas Urbanas e Industriales              797
Terrenos Agrícolas                          375
Áreas Urbanas e Industriales                359
Áreas no Reconocidas                          5
Name: count, dtype: int64


In [ ]:
# CELDA 6 — Corregir las categorías con encoding residual
mapeo_correccion = {
    'ÃƒÂreas Desprovistas de VegetaciÃƒÂ³n': 'Áreas Desprovistas de Vegetación',
    'Terrenos AgrÃƒÂ­colas': 'Terrenos Agrícolas',
    'ÃƒÂreas Urbanas e Industriales': 'Áreas Urbanas e Industriales',
}
uso_suelo_magallanes['USO'] = uso_suelo_magallanes['USO'].replace(mapeo_correccion)

print("Categorías USO finales:")
print(uso_suelo_magallanes['USO'].value_counts())

Categorías USO finales:
USO
Bosques                             111638
Praderas y Matorrales               106399
Cuerpos de Agua                      32091
Áreas Desprovistas de Vegetación     18421
Humedales                            14432
Nieves Eternas y Glaciares            1205
Terrenos Agrícolas                    1182
Áreas Urbanas e Industriales          1156
Áreas no Reconocidas                     5
Name: count, dtype: int64


In [ ]:
# CELDA 7 — Aplicar tabla de aptitud 0-5
tabla_aptitud = {
    'Áreas Desprovistas de Vegetación': 5,
    'Praderas y Matorrales':            4,
    'Terrenos Agrícolas':               2,
    'Bosques':                          1,
    'Humedales':                        0,
    'Cuerpos de Agua':                  0,
    'Áreas Urbanas e Industriales':     0,
    'Nieves Eternas y Glaciares':       0,
    'Áreas no Reconocidas':             0,
}
uso_suelo_magallanes['APTITUD_USO'] = uso_suelo_magallanes['USO'].map(tabla_aptitud)

sin_mapear = uso_suelo_magallanes['APTITUD_USO'].isna().sum()
print(f"Polígonos sin aptitud asignada: {sin_mapear}")
print(uso_suelo_magallanes['APTITUD_USO'].value_counts().sort_index())

Polígonos sin aptitud asignada: 0
APTITUD_USO
0     48889
1    111638
2      1182
4    106399
5     18421
Name: count, dtype: int64


In [ ]:
# CELDA 8 — Guardar de inmediato (apenas esté listo, antes de hacer nada más)
ruta_salida = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp'
uso_suelo_magallanes.to_file(ruta_salida)
print(f"💾 Guardado: {ruta_salida}")

/tmp/ipykernel_55955/3103903021.py:3: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  uso_suelo_magallanes.to_file(ruta_salida)
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'APTITUD_USO' to 'APTITUD_US'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: 2GB file size limit reached for /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp. Going on, but might cause compatibility issues with third party software
  ogr_write(


💾 Guardado: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp


In [ ]:
# ============================================================
# Rasterizar uso de suelo, usando la grilla de pendiente como referencia
# (misma resolución y extensión que pendiente_magallanes_v2.tif)
# ============================================================
import rasterio
from rasterio.features import rasterize
import numpy as np

carpeta = '/content/drive/MyDrive/GEO3211_Eolico'

# Usamos pendiente como "molde" de referencia (mismo CRS: EPSG:32719)
with rasterio.open(f'{carpeta}/pendiente_magallanes_v2.tif') as ref:
    perfil_ref = ref.profile.copy()
    shape_ref = (ref.height, ref.width)
    transform_ref = ref.transform

# Preparar pares (geometría, valor) para rasterizar
formas_valores = [
    (geom, valor)
    for geom, valor in zip(uso_suelo_magallanes.geometry, uso_suelo_magallanes['APTITUD_USO'])
]

uso_suelo_raster = rasterize(
    formas_valores,
    out_shape=shape_ref,
    transform=transform_ref,
    fill=-9999,        # valor para área sin dato (debería ser mínimo, fuera de Magallanes)
    dtype='float32'
)

print(f"✅ Raster generado: {uso_suelo_raster.shape}")
print(f"Valores únicos: {np.unique(uso_suelo_raster)}")

NameError: name 'uso_suelo_magallanes' is not defined

In [ ]:
# CELDA 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELDA 2 — Verificar RAM
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       845Mi       3.2Gi       2.0Mi       8.6Gi        11Gi
Swap:             0B          0B          0B


In [ ]:
# CELDA 3 — Cargar el shapefile YA UNIFICADO (mucho más liviano que antes)
import geopandas as gpd

carpeta = '/content/drive/MyDrive/GEO3211_Eolico'
uso_suelo_magallanes = gpd.read_file(f'{carpeta}/uso_suelo_magallanes_unificado.shp')

print(f"Polígonos cargados: {len(uso_suelo_magallanes)}")
print(uso_suelo_magallanes.columns.tolist())

Polígonos cargados: 286529
['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'APTITUD_US', 'geometry']


In [ ]:
# CELDA 4 — Confirmar que la columna APTITUD_USO sigue ahí
print(uso_suelo_magallanes['APTITUD_USO'].value_counts().sort_index())

KeyError: 'APTITUD_USO'

In [ ]:
# ============================================================
# Regenerar la columna APTITUD_USO (no se guardó en el shapefile anterior)
# ============================================================
tabla_aptitud = {
    'Áreas Desprovistas de Vegetación': 5,
    'Praderas y Matorrales':            4,
    'Terrenos Agrícolas':               2,
    'Bosques':                          1,
    'Humedales':                        0,
    'Cuerpos de Agua':                  0,
    'Áreas Urbanas e Industriales':     0,
    'Nieves Eternas y Glaciares':       0,
    'Áreas no Reconocidas':             0,
}

uso_suelo_magallanes['APTITUD_USO'] = uso_suelo_magallanes['USO'].map(tabla_aptitud)

sin_mapear = uso_suelo_magallanes['APTITUD_USO'].isna().sum()
print(f"Polígonos sin aptitud asignada: {sin_mapear}")
print(uso_suelo_magallanes['APTITUD_USO'].value_counts().sort_index())

Polígonos sin aptitud asignada: 0
APTITUD_USO
0     48889
1    111638
2      1182
4    106399
5     18421
Name: count, dtype: int64


In [ ]:
# ============================================================
# Sobrescribir el shapefile, esta vez incluyendo APTITUD_USO
# ============================================================
ruta_salida = '/content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp'
uso_suelo_magallanes.to_file(ruta_salida)
print(f"💾 Actualizado con columna de aptitud: {ruta_salida}")

/tmp/ipykernel_60904/1902808502.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  uso_suelo_magallanes.to_file(ruta_salida)
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'APTITUD_USO' to 'APTITUD__1'
  ogr_write(
/usr/local/lib/python3.12/dist-packages/pyogrio/raw.py:733: RuntimeWarning: 2GB file size limit reached for /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp. Going on, but might cause compatibility issues with third party software
  ogr_write(


💾 Actualizado con columna de aptitud: /content/drive/MyDrive/GEO3211_Eolico/uso_suelo_magallanes_unificado.shp


In [ ]:
tabla_aptitud = {
    'Áreas Desprovistas de Vegetación': 5,
    'Praderas y Matorrales': 4,
    'Terrenos Agrícolas': 2,
    'Bosques': 1,
    'Humedales': 0,
    'Cuerpos de Agua': 0,
    'Áreas Urbanas e Industriales': 0,
    'Nieves Eternas y Glaciares': 0,
    'Áreas no Reconocidas': 0,
}
uso_suelo_magallanes['APTITUD_USO'] = uso_suelo_magallanes['USO'].map(tabla_aptitud)

In [ ]:
# ============================================================
# Rasterizar uso de suelo, usando pendiente como grilla de referencia
# ============================================================
import rasterio
from rasterio.features import rasterize
import numpy as np

carpeta = '/content/drive/MyDrive/GEO3211_Eolico'

with rasterio.open(f'{carpeta}/pendiente_magallanes_v2.tif') as ref:
    perfil_ref = ref.profile.copy()
    shape_ref = (ref.height, ref.width)
    transform_ref = ref.transform

formas_valores = [
    (geom, valor)
    for geom, valor in zip(uso_suelo_magallanes.geometry, uso_suelo_magallanes['APTITUD_USO'])
]

uso_suelo_raster = rasterize(
    formas_valores,
    out_shape=shape_ref,
    transform=transform_ref,
    fill=-9999,
    dtype='float32'
)

print(f"✅ Raster generado: {uso_suelo_raster.shape}")
print(f"Valores únicos: {np.unique(uso_suelo_raster)}")

Aquí ocurrió un fallo de RAM.

In [ ]:
# CELDA 1 — Montar Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# CELDA 2 — Verificar RAM (confirma que tienes margen antes de seguir)
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       1.1Gi        10Gi       2.0Mi       737Mi        11Gi
Swap:             0B          0B          0B


In [ ]:
# CELDA 3 — Cargar SOLO lo que se necesita para rasterizar
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
import numpy as np

carpeta = '/content/drive/MyDrive/GEO3211_Eolico'

uso_suelo_magallanes = gpd.read_file(f'{carpeta}/uso_suelo_magallanes_unificado.shp')

# el shapefile truncó el nombre a 10 caracteres, revisemos cuál quedó
print(uso_suelo_magallanes.columns.tolist())

['USO', 'SUBUSO', 'COBERTURA', 'SUPERF_HA', 'TIPO_SNASP', 'APTITUD_US', 'APTITUD__1', 'geometry']


In [ ]:
# ============================================================
# Verificar cuál columna tiene los valores correctos de aptitud
# ============================================================
print("APTITUD_US:")
print(uso_suelo_magallanes['APTITUD_US'].value_counts(dropna=False).sort_index())

print("\nAPTITUD__1:")
print(uso_suelo_magallanes['APTITUD__1'].value_counts(dropna=False).sort_index())

APTITUD_US:
APTITUD_US
0     48889
1    111638
2      1182
4    106399
5     18421
Name: count, dtype: int64

APTITUD__1:
APTITUD__1
0     48889
1    111638
2      1182
4    106399
5     18421
Name: count, dtype: int64


In [ ]:
# ============================================================
# Rasterizar uso de suelo usando pendiente como grilla de referencia
# ============================================================
with rasterio.open(f'{carpeta}/pendiente_magallanes_v2.tif') as ref:
    perfil_ref = ref.profile.copy()
    shape_ref = (ref.height, ref.width)
    transform_ref = ref.transform

formas_valores = [
    (geom, valor)
    for geom, valor in zip(uso_suelo_magallanes.geometry, uso_suelo_magallanes['APTITUD_US'])
]

uso_suelo_raster = rasterize(
    formas_valores,
    out_shape=shape_ref,
    transform=transform_ref,
    fill=-9999,
    dtype='float32'
)

print(f"✅ Raster generado: {uso_suelo_raster.shape}")

In [ ]:
# ============================================================
# Guardar de inmediato
# ============================================================
perfil_uso_suelo = perfil_ref.copy()
perfil_uso_suelo.update({'nodata': -9999, 'dtype': 'float32'})

ruta_raster_uso = f'{carpeta}/aptitud_uso_suelo_magallanes.tif'
with rasterio.open(ruta_raster_uso, 'w', **perfil_uso_suelo) as dst:
    dst.write(uso_suelo_raster, 1)

print(f"💾 Guardado: {ruta_raster_uso}")

In [ ]:
# ============================================================
# Diagnóstico: ¿qué tamaño tiene la grilla que intentamos generar?
# ============================================================
import rasterio

carpeta = '/content/drive/MyDrive/GEO3211_Eolico'

with rasterio.open(f'{carpeta}/pendiente_magallanes_v2.tif') as ref:
    print(f"Dimensiones: {ref.height} x {ref.width}")
    print(f"Total de píxeles: {ref.height * ref.width:,}")
    print(f"Tamaño estimado en RAM (float32): {(ref.height * ref.width * 4) / (1024**2):.1f} MB")

Dimensiones: 8868 x 7263
Total de píxeles: 64,408,284
Tamaño estimado en RAM (float32): 245.7 MB


In [ ]:
# ============================================================
# Simplificar geometrías antes de rasterizar (reduce vértices)
# ============================================================
uso_suelo_magallanes['geometry'] = uso_suelo_magallanes.geometry.simplify(
    tolerance=50,  # 50 metros de tolerancia, razonable para pixeles de 90m
    preserve_topology=True
)

print("✅ Geometrías simplificadas")

NameError: name 'uso_suelo_magallanes' is not defined